# Training Data Exploration

Explore the parquet training table extracted for the autoencoder.

In [1]:
from pathlib import Path
import pandas as pd

# Adjust popularity threshold as needed
POPULARITY = 80

data_path = Path("..") / "data" / f"popularity{POPULARITY}" / "tracks.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} tracks from {data_path}")

Loaded 1,640 tracks from ../data/popularity80/tracks.parquet


## Basic Info

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1640 entries, 0 to 1639
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   track_rowid             1640 non-null   int64  
 1   track_id                1640 non-null   object 
 2   track_name              1640 non-null   object 
 3   album_rowid             1640 non-null   int64  
 4   external_id_isrc        1640 non-null   object 
 5   track_popularity        1640 non-null   int64  
 6   track_number            1640 non-null   int64  
 7   disc_number             1640 non-null   int64  
 8   duration_ms             1640 non-null   int64  
 9   explicit                1640 non-null   int64  
 10  artist_rowid            1640 non-null   int64  
 11  artist_name             1640 non-null   object 
 12  artist_popularity       1640 non-null   int64  
 13  artist_followers        1640 non-null   int64  
 14  genres                  1640 non-null   

In [3]:
df.head(10)

,track_rowid,track_id,track_name,album_rowid,external_id_isrc,track_popularity,track_number,disc_number,duration_ms,explicit,...,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,273,1LLUoftvmTjVNBHZoQyveF,THE DINER,102,USUM72401989,80,8,1,186346,0,...,1,1,0.857,0.386,-9.761,0.1680,0.24300,0.093100,0.1110,0.6610
1,461,3OHfY25tqY28d16oZczHc8,Kill Bill,119,USRC12204584,80,2,1,153946,0,...,8,1,0.644,0.728,-5.750,0.0351,0.05430,0.169000,0.1610,0.4300
2,463,2GAhgAjOhEmItWLfgisyOn,Low,119,USRC12204586,80,4,1,181080,1,...,0,0,0.698,0.546,-7.234,0.0559,0.42500,0.003140,0.1600,0.3350
3,534,1MgV7FIyNxIG7WzMRJV5HC,my tears ricochet,122,USUG12002839,80,5,1,255893,0,...,0,1,0.469,0.263,-10.630,0.0329,0.80600,0.000000,0.0749,0.1120
4,589,3IX0yuEVvDbnqUwMBB3ouC,bad idea right?,126,USUG12304094,80,2,1,184783,1,...,9,1,0.631,0.881,-3.545,0.0995,0.00177,0.000005,0.0639,0.8080
5,591,6QT6j7rKt7Vk3IuV2AUO9W,lacy,126,USUG12304095,80,4,1,177212,0,...,3,1,0.395,0.367,-7.653,0.0324,0.81500,0.000000,0.1100,0.4260
6,604,3Vi5XqYrmQgOYBajMWSvCi,Need to Know,127,USRC12101120,80,5,1,210560,1,...,1,1,0.664,0.609,-6.509,0.0707,0.30400,0.000000,0.0926,0.1940
7,631,6NFyWDv5CjfwuzoCkw47Xf,Delicate,129,USCJY1750007,80,5,1,232253,0,...,9,0,0.750,0.404,-10.178,0.0682,0.21600,0.000357,0.0911,0.0499
8,632,1P17dC1amhFzptugyAO7Il,Look What You Made Me Do,129,USCJY1750000,80,6,1,211853,0,...,9,0,0.766,0.709,-6.471,0.1230,0.20400,0.000014,0.1260,0.5060
9,713,0uoWOZeZZiC90RMNDQBqj4,Don't Call Tonight,134,USUM72412589,80,10,1,225804,0,...,0,1,0.652,0.865,-3.487,0.0457,0.00546,0.000000,0.3360,0.7380


## Audio Features Statistics

In [ ]:
audio_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "time_signature", "key", "mode"
]

df[audio_features].describe()

## Popularity & Metadata Statistics

In [ ]:
metadata_cols = [
    "track_popularity", "album_popularity", "artist_popularity",
    "artist_followers", "duration_ms"
]

df[metadata_cols].describe()

## Categorical Columns

In [ ]:
print("Album types:")
print(df["album_type"].value_counts())
print()
print("Explicit:")
print(df["explicit"].value_counts())
print()
print("Mode (0=minor, 1=major):")
print(df["mode"].value_counts())

## Genre Analysis

In [ ]:
# Tracks without genres
no_genre = df["genres"].isna() | (df["genres"] == "")
print(f"Tracks without genres: {no_genre.sum():,} ({no_genre.mean()*100:.1f}%)")

In [ ]:
# Most common genres
all_genres = df["genres"].dropna().str.split(";").explode()
print(f"Total genre tags: {len(all_genres):,}")
print(f"Unique genres: {all_genres.nunique():,}")
print()
print("Top 20 genres:")
all_genres.value_counts().head(20)

## Missing Values

In [ ]:
missing = df.isna().sum()
missing[missing > 0]

## Top Artists by Track Count

In [ ]:
df.groupby("artist_name").size().sort_values(ascending=False).head(20)

## Release Date Distribution

In [ ]:
# Extract year from release_date
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["release_year"].value_counts().sort_index().tail(20)

## Correlation Matrix (Audio Features)

In [ ]:
df[audio_features].corr().round(2)